In [1]:
import os
import glob
import pathlib
import re
import pandas as pd

In [2]:
core_path = os.getcwd()
spectra_filepaths = sorted(glob.glob(os.path.join(core_path, "data/lvl0/", "smass2/*spfit*")))  #we are finding all the files inside the smass2 whose name contains "spfit"

In [3]:
des_file_paths = spectra_filepaths[:-8] # taking all except last 8 for designated
non_file_paths = spectra_filepaths[-8:] # taking last 8 except all for non designated

des_file_paths_df = pd.DataFrame(des_file_paths, columns=["FilePath"])  # converting it into a dataframe with File Paths as a column
non_file_paths_df = pd.DataFrame(non_file_paths, columns=["FilePath"])

# des_file_paths_df.head()

print(des_file_paths_df.FilePath.iloc[0])
print(non_file_paths_df.FilePath.iloc[0])

/media/shade/DA86EB1886EAF443/Projects/Spectral and Dynamical Analysis of Asteroid Families/data/lvl0/smass2/a000001.spfit.[2]
/media/shade/DA86EB1886EAF443/Projects/Spectral and Dynamical Analysis of Asteroid Families/data/lvl0/smass2/au1995BM2.spfit.[2]


In [7]:
# for all rows, set the value in the colum DesNr to the result of the lambda function.

des_file_paths_df["DesNr"] = des_file_paths_df["FilePath"].apply(
    lambda x: int(m.group(1)) if (m := re.search(r"smass2[/\\]a(.*?)\.spfit", str(x))) else None
)

non_file_paths_df["DesNr"] = non_file_paths_df["FilePath"].apply(
    lambda x: m.group(1) if (m := re.search(r"smass2[/\\]au(.*?)\.spfit", str(x))) else None
)

In [8]:
non_file_paths_df.head()

,FilePath,DesNr
0,/media/shade/DA86EB1886EAF443/Projects/Spectra...,1995BM2
1,/media/shade/DA86EB1886EAF443/Projects/Spectra...,1995WQ5
2,/media/shade/DA86EB1886EAF443/Projects/Spectra...,1996PW
3,/media/shade/DA86EB1886EAF443/Projects/Spectra...,1996UK
4,/media/shade/DA86EB1886EAF443/Projects/Spectra...,1996VC


In [9]:
des_file_paths_df.head()

,FilePath,DesNr
0,/media/shade/DA86EB1886EAF443/Projects/Spectra...,1
1,/media/shade/DA86EB1886EAF443/Projects/Spectra...,2
2,/media/shade/DA86EB1886EAF443/Projects/Spectra...,3
3,/media/shade/DA86EB1886EAF443/Projects/Spectra...,4
4,/media/shade/DA86EB1886EAF443/Projects/Spectra...,5


In [10]:
# Reading the classification files

asteroid_class_df = pd.read_csv(os.path.join(core_path, "data/lvl0/", "Bus.Taxonomy.txt"),
                                skiprows=21,     # skipping 21 lines from the file (worthless)
                                sep="\t",        # seperate on tab space
                                names=["Name",
                                       "Tholen Class",
                                       "Bus Class",
                                       "unknown1",
                                       "unknown2"] # Unknown fields were added because 3 fields gives tokenization errors, where it counts 5. 
                                )

asteroid_class_df.head(5)
# asteroid_class_df[-8:]

,Name,Tholen Class,Bus Class,unknown1,unknown2
0,1 Ceres,G,C,NaN,NaN
1,2 Pallas,B,B,NaN,NaN
2,3 Juno,S,Sk,NaN,NaN
3,4 Vesta,V,V,NaN,NaN
4,5 Astraea,S,S,NaN,NaN


In [11]:
# removing the white spaces

asteroid_class_df.loc[:, "Name"] = asteroid_class_df["Name"].apply(lambda x: x.strip()).copy()

# asteroid_class_df.head(5)
asteroid_class_df[-8:]

,Name,Tholen Class,Bus Class,unknown1,unknown2
1439,1998 VR,NaN,Sk,NaN,NaN
1440,1998 VO33,NaN,V,NaN,NaN
1441,1998 WM,NaN,Sq,NaN,NaN
1442,1998 WS,NaN,Sr,NaN,NaN
1443,1998 WZ6,NaN,V,NaN,NaN
1444,1999 EE5,NaN,S,NaN,NaN
1445,1999 FA,NaN,S,NaN,NaN
1446,1999 FB,NaN,Q,NaN,NaN


In [12]:
# seperate between designated and non-designated asteroid classes

des_ast_class_df = asteroid_class_df[:1403].copy()

non_ast_class_df = asteroid_class_df[1403:].copy()

In [13]:
# now we are splitting the designated names and getting the designated numbers (to link with spfit files)
des_ast_class_df.loc[:, "DesNr"] = des_ast_class_df["Name"].apply(lambda x: int(x.split(" ")[0]))
# des_ast_class_df[-8:]


# Merge with the spectral file paths

des_ast_class_join_df = des_ast_class_df.merge(des_file_paths_df, on="DesNr")
# des_ast_class_join_df.head()

# Merging the non-designated names, we need to remove the white space between number and the name and then compare with file paths.

non_ast_class_df.loc[:, "DesNr"] = non_ast_class_df["Name"].apply(lambda x: x.replace(" ", ""))
# non_ast_class_df.head()

# Merge with spectral file paths

non_ast_class_join_df = non_ast_class_df.merge(non_file_paths_df, on="DesNr")
non_ast_class_join_df.head()


,Name,Tholen Class,Bus Class,unknown1,unknown2,DesNr,FilePath
0,1995 BM2,NaN,Sq,NaN,NaN,1995BM2,/media/shade/DA86EB1886EAF443/Projects/Spectra...
1,1995 WQ5,NaN,Ch,NaN,NaN,1995WQ5,/media/shade/DA86EB1886EAF443/Projects/Spectra...
2,1996 PW,NaN,Ld,NaN,NaN,1996PW,/media/shade/DA86EB1886EAF443/Projects/Spectra...
3,1996 UK,NaN,Sq,NaN,NaN,1996UK,/media/shade/DA86EB1886EAF443/Projects/Spectra...
4,1996 VC,NaN,S,NaN,NaN,1996VC,/media/shade/DA86EB1886EAF443/Projects/Spectra...


In [14]:
# Merging both datasets that we created now
asteroids_df = pd.concat([des_ast_class_join_df, non_ast_class_join_df], axis=0)

asteroids_df.reset_index(drop=True, inplace=True)
asteroids_df.drop(columns=["Tholen Class", "unknown1", "unknown2"], inplace=True)
asteroids_df.dropna(subset=["Bus Class"], inplace=True)

In [15]:
# Read and Store the spectra into a dataframe

asteroids_df.loc[:, "SpectrumDF"] = asteroids_df["FilePath"].apply(lambda x: pd.read_csv(x, sep="\t",
                                                                                         names=["Wavelength_in_micron",
                                                                                                "Reflectance_norm550nm"]))

asteroids_df.reset_index(drop=True, inplace=True)

asteroids_df.loc[:, "DesNr"] = asteroids_df["DesNr"].astype(str)

In [16]:
asteroids_df.head(-8)

,Name,Bus Class,DesNr,FilePath,SpectrumDF
0,1 Ceres,C,1,/media/shade/DA86EB1886EAF443/Projects/Spectra...,Wavelength_in_micron Reflectance_norm550n...
1,2 Pallas,B,2,/media/shade/DA86EB1886EAF443/Projects/Spectra...,Wavelength_in_micron Reflectance_norm550n...
2,3 Juno,Sk,3,/media/shade/DA86EB1886EAF443/Projects/Spectra...,Wavelength_in_micron Reflectance_norm550n...
3,4 Vesta,V,4,/media/shade/DA86EB1886EAF443/Projects/Spectra...,Wavelength_in_micron Reflectance_norm550n...
4,5 Astraea,S,5,/media/shade/DA86EB1886EAF443/Projects/Spectra...,Wavelength_in_micron Reflectance_norm550n...
...,...,...,...,...,...
1326,11785 1973 AW3,Xc,11785,/media/shade/DA86EB1886EAF443/Projects/Spectra...,Wavelength_in_micron Reflectance_norm550n...
1327,11906 1992 AE1,Sq,11906,/media/shade/DA86EB1886EAF443/Projects/Spectra...,Wavelength_in_micron Reflectance_norm550n...
1328,12281 1990 WA5,X,12281,/media/shade/DA86EB1886EAF443/Projects/Spectra...,Wavelength_in_micron Reflectance_norm550n...
1329,17480 1991 PE10,S,17480,/media/shade/DA86EB1886EAF443/Projects/Spectra...,Wavelength_in_micron Reflectance_norm550n...


In [17]:
pathlib.Path(os.path.join(core_path, "data/lvl1")).mkdir(parents=True, exist_ok=True)

asteroids_df.to_pickle(os.path.join(core_path, "data/lvl1/", "asteroids_merged.pk1"), protocol=4)